# 

# HW 1 - Multiple Linear Regression

In [ ]:
## Problem 2: Model and Analyze a Real Dataset
# Predicting Flight Arrival Delay Using Multiple Linear Regression
# In this project, I use departure delay, taxi out time, taxi in time, air time, and distance to predict flight arrival delay.

In [ ]:

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

df = pd.read_csv("DatasetP2.csv")

df.head()
df.info()
df.columns

model_vars = ["ARR_DELAY", "DEP_DELAY", "TAXI_OUT", "TAXI_IN", "AIR_TIME", "DISTANCE"]

data = df[model_vars].dropna()

#data.head(5)

sns.pairplot(data, diag_kind="hist", plot_kws={"alpha": 0.5, "s": 15})
plt.suptitle("Pairplot of Flight Arrival Delay Dataset", y=1.02)
plt.show()

In [ ]:
## Scatterplot Matrix Discussion
# The scatterplot matrix shows that `DEP_DELAY` has a strong positive linear relationship with `ARR_DELAY`. This means flights that depart late are also likely to arrive late. `TAXI_OUT` and `TAXI_IN` also appear to have positive relationships with arrival delay, but the relationships are weaker than the relationship between departure delay and arrival delay.
# `AIR_TIME` and `DISTANCE` are strongly related because longer flights usually have longer air time and longer distance. This may cause some multicollinearity in the model. Overall, the scatterplot matrix suggests that these variables are reasonable predictors to include in the multiple linear regression model.

In [ ]:
formula = "ARR_DELAY ~ DEP_DELAY + TAXI_OUT + TAXI_IN + AIR_TIME + DISTANCE"
mlr_model = smf.ols(formula=formula, data=data).fit()

print(mlr_model.summary())

coefficients = pd.DataFrame({"Term": mlr_model.params.index, "Coefficient": mlr_model.params.values, "P-value": mlr_model.pvalues.values})
print(coefficients)

b0 = mlr_model.params["Intercept"]
b1 = mlr_model.params["DEP_DELAY"]
b2 = mlr_model.params["TAXI_OUT"]
b3 = mlr_model.params["TAXI_IN"]
b4 = mlr_model.params["AIR_TIME"]
b5 = mlr_model.params["DISTANCE"]

print(f"Estimated coefficients:\nIntercept: {b0:.4f}\nDEP_DELAY: {b1:.4f}\nTAXI_OUT: {b2:.4f}\nTAXI_IN: {b3:.4f}\nAIR_TIME: {b4:.4f}\nDISTANCE: {b5:.4f}")

anova_results = anova_lm(mlr_model)
print(anova_results)

data["PREDICTED_ARR_DELAY"] = mlr_model.predict(data)

# Actual vs. predicted plot

plt.figure(figsize=(8, 6))

plt.scatter(
    data["ARR_DELAY"],
    data["PREDICTED_ARR_DELAY"],
    alpha=0.4
)

min_val = min(
    data["ARR_DELAY"].min(),
    data["PREDICTED_ARR_DELAY"].min()
)

max_val = max(
    data["ARR_DELAY"].max(),
    data["PREDICTED_ARR_DELAY"].max()
)

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    color="red",
    linestyle="--",
    label="Perfect Prediction"
)

plt.xlabel("Actual Arrival Delay")
plt.ylabel("Predicted Arrival Delay")
plt.title("Actual vs. Predicted Arrival Delay")
plt.legend()
plt.show()

In [ ]:
## Conclusion
# The multiple linear regression model does a good job predicting `ARR_DELAY`. The \(R^2\) value is 0.958, which means the model explains about 95.8% of the variation in arrival delay. The overall F-test is significant, so at least one predictor is useful for predicting arrival delay.
# Among the predictors, `DEP_DELAY` is the most important variable. Its coefficient is 0.9944, meaning that each additional minute of departure delay increases predicted arrival delay by about 0.99 minutes. `TAXI_OUT`, `TAXI_IN`, and `AIR_TIME` also have positive effects on arrival delay, while `DISTANCE` has a small negative effect.
# Overall, the model is statistically significant and provides a strong fit. However, the high condition number suggests that there may be multicollinearity, especially between variables such as `AIR_TIME` and `DISTANCE`, so the coefficients should be interpreted carefully.